# Error Gating: does it induce a blocked preference?


In [ ]:

import sys
import types
from pathlib import Path

# Work around a Python 3.13 torch import crash in this conda env.
if 'rlcompleter' not in sys.modules:
    rlcompleter = types.ModuleType('rlcompleter')
    rlcompleter.Completer = type(
        'Completer',
        (),
        {
            '__init__': lambda self, namespace=None: None,
            'complete': lambda self, text, state: None,
        },
    )
    sys.modules['rlcompleter'] = rlcompleter

import pandas as pd
import numpy as np
from experimental_setups import error_gating_model, isc_model
import data
import torch
import torch.nn as nn
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

from experimental_setups.error_gating_model import *


## Updated Error-Gating Experiment


In [ ]:

results_path = Path('data/error_baseline_comparison.csv')
lc_path = Path('data/error_learning_curves.csv')
plot_dir = Path('images/error_gating')
plot_dir.mkdir(parents=True, exist_ok=True)

force_rerun = False
num_epochs = 40

needs_rerun = (
    force_rerun
    or not results_path.exists()
    or not lc_path.exists()
)

if not needs_rerun:
    lc_probe = pd.read_csv(lc_path, nrows=1)
    needs_rerun = not {'c1', 'c2'}.issubset(lc_probe.columns)

if needs_rerun:
    print('Now training error-gating model...')
    hebb_errors, hebb_lc = run_error_experiment(
        num_epochs=num_epochs,
        track_learning_curves=True,
        save_models=True,
    )
    print('	...finished error-gating model!')

    print('Now training/loading ISC baseline...')
    error_data = hebb_errors + run_error_baseline(num_epochs=num_epochs)
    error_data = pd.concat(error_data, axis=0, ignore_index=True)
    error_data.to_csv(results_path, index=False)

    lc_df = pd.concat(hebb_lc, axis=0, ignore_index=True)
    lc_df.to_csv(lc_path, index=False)
    print('	...finished baseline and saved CSVs!')
else:
    print('Using cached experiment CSVs.')

lc_df = pd.read_csv(lc_path)
results_df = pd.read_csv(results_path)

lc_df['epoch'] = lc_df['epoch'].astype(int)
lc_df['bce'] = lc_df['bce'].astype(float)
lc_df[['c1', 'c2']] = lc_df[['c1', 'c2']].astype(float)

print(lc_df.head())
print(results_df.head())


## Learning Curves and Context-Input-to-CD Weights


In [ ]:

lc_summary = lc_df.groupby(['condition', 'epoch'], as_index=False).agg(
    bce_mean=('bce', 'mean'),
    bce_sem=('bce', lambda x: x.std(ddof=1) / np.sqrt(len(x))),
    c1_mean=('c1', 'mean'),
    c1_sem=('c1', lambda x: x.std(ddof=1) / np.sqrt(len(x))),
    c2_mean=('c2', 'mean'),
    c2_sem=('c2', lambda x: x.std(ddof=1) / np.sqrt(len(x))),
)

colors = {'interleaved': '#2563eb', 'blocked': '#dc2626'}

fig, ax = plt.subplots(figsize=(7, 4.5))
for condition, condition_df in lc_summary.groupby('condition'):
    ax.plot(
        condition_df['epoch'],
        condition_df['bce_mean'],
        label=condition,
        color=colors.get(condition),
    )
    ax.fill_between(
        condition_df['epoch'],
        condition_df['bce_mean'] - condition_df['bce_sem'],
        condition_df['bce_mean'] + condition_df['bce_sem'],
        color=colors.get(condition),
        alpha=0.18,
    )

ax.set_xlabel('Epoch')
ax.set_ylabel('BCE')
ax.set_title('Error-gating learning curves')
ax.spines[['top', 'right']].set_visible(False)
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(plot_dir / 'error_learning_curves.png', dpi=200)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True)
for ax, weight in zip(axes, ['c1', 'c2']):
    for condition, condition_df in lc_summary.groupby('condition'):
        mean = condition_df[f'{weight}_mean']
        sem = condition_df[f'{weight}_sem']
        ax.plot(condition_df['epoch'], mean, label=condition, color=colors.get(condition))
        ax.fill_between(
            condition_df['epoch'],
            mean - sem,
            mean + sem,
            color=colors.get(condition),
            alpha=0.18,
        )
    ax.set_title(weight)
    ax.set_xlabel('Epoch')
    ax.spines[['top', 'right']].set_visible(False)

axes[0].set_ylabel('Context-input-to-CD weight')
axes[1].legend(frameon=False)
fig.suptitle('Tracked context weights')
fig.tight_layout()
fig.savefig(plot_dir / 'context_weight_traces.png', dpi=200)
plt.show()


## Error-Gating vs ISC Baseline


In [ ]:

model_means = results_df.groupby(['architecture', 'condition', 'model'])['acc'].mean()
stats = model_means.groupby(['architecture', 'condition']).agg(['mean', 'std'])

plot_data = stats['mean'].unstack('condition').loc[['Hebbian', 'ISC']]
plot_err = stats['std'].unstack('condition').loc[['Hebbian', 'ISC']]
plot_data.index = ['Error Gating', 'ISC']
plot_err.index = plot_data.index

fig, ax = plt.subplots(figsize=(6.5, 4.5))
plot_data[['blocked', 'interleaved']].plot(
    kind='bar',
    yerr=plot_err[['blocked', 'interleaved']],
    capsize=5,
    ax=ax,
    color=[colors['blocked'], colors['interleaved']],
)

ax.set_xlabel('Architecture')
ax.set_ylabel('Accuracy')
ax.set_ylim(0, 1)
ax.set_title('Error Gating vs Baseline: Blocked vs Interleaved')
ax.spines[['top', 'right']].set_visible(False)
ax.legend(title='', frameon=False)
fig.tight_layout()
fig.savefig(plot_dir / 'error_baseline_comparison.png', dpi=200)
plt.show()
